#### 05. Tier 1 features

Tier 1 describes the rain at the station itself up to and including the prediction time: totals over the last 5 minutes to 24 hours, the strongest 5-minute reading of the last half hour, how many of the last 12 readings were wet, whether the rain is building or easing, how long ago it last rained, and how many readings are missing in the last 2 hours. Every feature uses only readings at or before the prediction time, so nothing from the label window leaks in. Missing readings are skipped, not counted as zero, and `t1_missing_120` tells the model how complete the recent history is.

The features are computed once for every reading on the cleaned 5-minute grid and saved to `task2/data/feat_t1/`; notebook 03 joins them into the train, validation and test tables.

**Setup**

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../code")
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import config as C
import features as FT

spark = C.get_spark("task2-tier1", cores=8, memory="10g")

**Building the features**

In [2]:
grid = FT.clean_grid(spark)
FT.add_tier1(grid).write.mode("overwrite").partitionBy("year").parquet(str(C.FEAT_T1))

feat = spark.read.parquet(str(C.FEAT_T1))
T1 = [c for c in feat.columns if c.startswith("t1_")]
print(f"{feat.count():,} rows, {len(T1)} features:", T1)

48,448,514 rows, 11 features: ['t1_rain_5', 't1_rain_15', 't1_rain_30', 't1_rain_60', 't1_rain_120', 't1_rain_1440', 't1_max5_30', 't1_wet_slots_60', 't1_trend_30', 't1_min_since_rain', 't1_missing_120']


In [3]:
feat.filter((F.col("station_id") == "S07") & F.col("slot").between(660864, 660872)).orderBy("slot").select("slot", *T1).toPandas()

,slot,t1_rain_5,t1_rain_15,t1_rain_30,t1_rain_60,t1_rain_120,t1_rain_1440,t1_max5_30,t1_wet_slots_60,t1_trend_30,t1_min_since_rain,t1_missing_120
0,660864,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,1440,0
1,660865,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,1440,0
2,660866,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,1440,0
3,660867,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1,1.4,0,0
4,660868,1.4,2.8,2.8,2.8,2.8,2.8,1.4,2,2.8,0,0
5,660869,0.2,3.0,3.0,3.0,3.0,3.0,1.4,3,3.0,0,0
6,660870,0.2,1.8,3.2,3.2,3.2,3.2,1.4,4,0.4,0,0
7,660871,0.0,0.4,3.2,3.2,3.2,3.2,1.4,4,-2.4,5,0
8,660872,0.0,0.2,3.2,3.2,3.2,3.2,1.4,4,-2.8,10,0


**Which features carry signal**

Each feature on its own is used to rank a 20 % sample of the training rows (2017–2022) by the chance of rain in the next 30 minutes, scored with AUC-PR, both on all rows and on the rows where it is dry now. A feature that cannot rank scores the event rate. The direction column says whether more of the feature means more rain (+) or less (−).

In [4]:
sample = (spark.read.parquet(str(C.LABELS))
               .filter(F.col("year").between(*C.TRAIN_YEARS))
               .sample(fraction=0.2, seed=C.SEED)
               .join(feat, ["station_id", "slot", "year"])
               .withColumn("wet", (F.col("label") > 0).cast("double"))
               .select("wet", "regime", *[F.col(c).cast("double") for c in T1])
               .cache())
subsets = {"all": sample, "dry now": sample.filter(F.col("regime") == "dry_now")}

means = sample.groupBy("wet").agg(*[F.avg(c).alias(c) for c in T1]).toPandas().set_index("wet")
direction = {c: 1 if means.loc[1.0, c] >= means.loc[0.0, c] else -1 for c in T1}
ev = BinaryClassificationEvaluator(labelCol="wet", metricName="areaUnderPR")

rows = []
for c in T1:
    row = {"feature": c, "direction": "+" if direction[c] > 0 else "−"}
    for name, df in subsets.items():
        row[f"AUC-PR {name}"] = ev.setRawPredictionCol("s").evaluate(df.withColumn("s", F.col(c) * direction[c]))
    rows.append(row)
rates = {f"AUC-PR {name}": df.agg(F.avg("wet")).first()[0] for name, df in subsets.items()}
signal = pd.DataFrame(rows + [{"feature": "event rate (no skill)", "direction": "", **rates}]).set_index("feature")
signal.round(4)

,direction,AUC-PR all,AUC-PR dry now
feature,,,
t1_rain_5,+,0.6578,0.0239
t1_rain_15,+,0.6739,0.0239
t1_rain_30,+,0.6477,0.0239
t1_rain_60,+,0.5883,0.1112
t1_rain_120,+,0.4988,0.0933
t1_rain_1440,+,0.1498,0.0365
t1_max5_30,+,0.6245,0.0239
t1_wet_slots_60,+,0.6140,0.1128
t1_trend_30,+,0.4519,0.0239


In [5]:
since = F.col("t1_min_since_rain")
bucket = (F.when(since == 0, "0 raining now").when(since <= 30, "1 within 30 min").when(since <= 60, "2 30–60 min")
           .when(since <= 120, "3 1–2 h").when(since <= 360, "4 2–6 h").when(since < 1440, "5 6–24 h")
           .otherwise("6 none in 24 h"))
(sample.groupBy(bucket.alias("last rain")).agg(F.count("*").alias("rows"), F.avg("wet").alias("P(rain in next 30 min)"))
       .orderBy("last rain").toPandas().set_index("last rain").round(4))

,rows,P(rain in next 30 min)
last rain,,
0 raining now,197292,0.8529
1 within 30 min,245612,0.3965
2 30–60 min,147977,0.1481
3 1–2 h,240647,0.0793
4 2–6 h,744215,0.0344
5 6–24 h,2217392,0.0194
6 none in 24 h,3101528,0.0127


Two groups of features show up. The short-window ones (rain in the last 5 to 30 minutes, the strongest reading, the trend) rank rainy moments well overall, with AUC-PR around 0.65 against an event rate of 0.06, but when it is dry now they are all zero and carry no information. For those onsets the useful features reach further back: rain in the last 60 or 120 minutes, the number of wet readings in the last hour and the time since the last rain lift AUC-PR from 0.024 to between 0.08 and 0.11, four to five times the event rate. The 24-hour total and the count of missing readings add little on their own.

The last table shows how fast persistence fades: the chance of rain in the next 30 minutes falls from 85 % while it is raining to 40 % within half an hour of the last rain, 15 % after 30 to 60 minutes and about 1 % after a dry day.